# BibTeX → Word XML bibliography converter

Converts `.bib` files into the Open XML bibliography format used by Word's **Manage Sources** feature (`Sources.xml`), so references can be imported into Word and cited with its built-in citation tool.

**How to use the output in Word:**
1. Run the cells below to produce an XML file (default: `word_sources.xml`).
2. Close Word, then copy the generated file over your master source list:
   - Windows: `%APPDATA%\Microsoft\Bibliography\Sources.xml`
   - macOS: `~/Library/Application Support/Microsoft/Office/Sources.xml`
3. Re-open Word → References → Manage Sources. All entries will appear in the *Master List* and can be inserted as citations / bibliography.

Only `pybtex` is required (already available via `sphinxcontrib-bibtex`), no extra install needed.

In [ ]:
import codecs
import re
import uuid
from pathlib import Path
from xml.sax.saxutils import escape

import latexcodec  # noqa: F401  (registers the 'ulatex' codec used to decode LaTeX accents)
from pybtex.database import parse_file

BIB_NAMESPACE = "http://schemas.openxmlformats.org/officeDocument/2006/bibliography"

# BibTeX entry type -> Word b:SourceType
ENTRY_TYPE_MAP = {
    "article": "JournalArticle",
    "book": "Book",
    "inbook": "BookSection",
    "incollection": "BookSection",
    "inproceedings": "ConferenceProceedings",
    "conference": "ConferenceProceedings",
    "proceedings": "ConferenceProceedings",
    "techreport": "Report",
    "manual": "Report",
    "mastersthesis": "Report",
    "phdthesis": "Report",
    "unpublished": "Misc",
    "online": "InternetSite",
    "electronic": "InternetSite",
    "misc": "Misc",
}
DEFAULT_SOURCE_TYPE = "Misc"

# BibTeX field -> Word b:* element, common to (almost) every source type
COMMON_FIELD_MAP = {
    "publisher": "Publisher",
    "address": "City",
    "edition": "Edition",
    "volume": "Volume",
    "number": "Issue",
    "pages": "Pages",
    "url": "URL",
}

# extra field mappings that only make sense for certain source types
TYPE_SPECIFIC_FIELD_MAP = {
    "JournalArticle": {"journal": "JournalName"},
    "ConferenceProceedings": {"booktitle": "ConferenceName"},
    "BookSection": {"booktitle": "BookTitle", "chapter": "ChapterNumber"},
    "Report": {
        "institution": "Institution",
        "school": "Institution",
        "organization": "Institution",
        "type": "Type",
    },
    "InternetSite": {"organization": "ProductionCompany"},
}

In [ ]:
def _clean(text: str) -> str:
    """Decode LaTeX accents/escapes to Unicode, strip leftover braces, collapse whitespace."""
    try:
        text = codecs.decode(text, "ulatex")
    except (ValueError, UnicodeDecodeError):
        pass
    text = re.sub(r"[{}]", "", text)
    text = text.replace("--", "-")
    return re.sub(r"\s+", " ", text).strip()


def _sanitize_tag(key: str) -> str:
    tag = re.sub(r"[^A-Za-z0-9]", "", key)
    return tag or uuid.uuid4().hex[:8]


def _person_parts(person):
    first = _clean(" ".join(person.first_names + person.middle_names))
    last = _clean(" ".join(person.prelast_names + person.last_names))
    return last, first


def _persons_xml(role: str, persons) -> str:
    people = []
    for person in persons:
        last, first = _person_parts(person)
        fields = [f"<b:Last>{escape(last)}</b:Last>"]
        if first:
            fields.append(f"<b:First>{escape(first)}</b:First>")
        people.append("<b:Person>" + "".join(fields) + "</b:Person>")
    namelist = "<b:NameList>" + "".join(people) + "</b:NameList>"
    return f"<b:{role}>{namelist}</b:{role}>"


def entry_to_source_xml(key: str, entry) -> str:
    fields = {name.lower(): value for name, value in entry.fields.items()}
    entry_type = entry.type.lower()
    source_type = ENTRY_TYPE_MAP.get(entry_type, DEFAULT_SOURCE_TYPE)
    if source_type == "Misc" and fields.get("url"):
        source_type = "InternetSite"

    parts = [
        f"<b:Tag>{escape(_sanitize_tag(key))}</b:Tag>",
        f"<b:SourceType>{source_type}</b:SourceType>",
        f"<b:Guid>{{{uuid.uuid4()}}}</b:Guid>",
    ]

    if "title" in fields:
        parts.append(f"<b:Title>{escape(_clean(fields['title']))}</b:Title>")
    if "year" in fields:
        parts.append(f"<b:Year>{escape(_clean(fields['year']))}</b:Year>")
    if "month" in fields:
        parts.append(f"<b:Month>{escape(_clean(fields['month']))}</b:Month>")

    roles_xml = []
    if entry.persons.get("author"):
        roles_xml.append(_persons_xml("Author", entry.persons["author"]))
    if entry.persons.get("editor"):
        roles_xml.append(_persons_xml("Editor", entry.persons["editor"]))
    if roles_xml:
        parts.append("<b:Author>" + "".join(roles_xml) + "</b:Author>")

    for bib_field, word_field in COMMON_FIELD_MAP.items():
        if bib_field in fields:
            parts.append(
                f"<b:{word_field}>{escape(_clean(fields[bib_field]))}</b:{word_field}>"
            )

    for bib_field, word_field in TYPE_SPECIFIC_FIELD_MAP.get(source_type, {}).items():
        if bib_field in fields:
            parts.append(
                f"<b:{word_field}>{escape(_clean(fields[bib_field]))}</b:{word_field}>"
            )

    comments = fields.get("note", "")
    if "doi" in fields:
        comments = (comments + f" DOI: {fields['doi']}").strip()
    if comments:
        parts.append(f"<b:Comments>{escape(_clean(comments))}</b:Comments>")

    return "<b:Source>" + "".join(parts) + "</b:Source>"


def bibtex_to_word_xml(bib_path, xml_path=None) -> str:
    """Convert a BibTeX file to a Word 2007+ Sources.xml bibliography.

    Args:
        bib_path: path to the input .bib file.
        xml_path: optional path to write the resulting XML to.

    Returns:
        The generated XML as a string.
    """
    bib_data = parse_file(str(bib_path), bib_format="bibtex")
    sources = [
        entry_to_source_xml(key, entry) for key, entry in bib_data.entries.items()
    ]

    xml = (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>\n'
        f'<b:Sources SelectedStyle="" xmlns:b="{BIB_NAMESPACE}" xmlns="{BIB_NAMESPACE}">\n  '
        + "\n  ".join(sources)
        + "\n</b:Sources>\n"
    )

    if xml_path is not None:
        Path(xml_path).write_text(xml, encoding="utf-8")

    return xml

In [ ]:
# Example: convert docs/references.bib to a Word-importable Sources.xml
BIB_PATH = "references.bib"
XML_PATH = "references_autogenerated.xml"

xml_text = bibtex_to_word_xml(BIB_PATH, XML_PATH)
print(f"Wrote {XML_PATH}")
print(xml_text[:1000])

Wrote references_autogenerated.xml
<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<b:Sources SelectedStyle="" xmlns:b="http://schemas.openxmlformats.org/officeDocument/2006/bibliography" xmlns="http://schemas.openxmlformats.org/officeDocument/2006/bibliography">
  <b:Source><b:Tag>aznarsiguan2019climada</b:Tag><b:SourceType>JournalArticle</b:SourceType><b:Guid>{be4e635a-0179-4761-bf4a-9985e20573a4}</b:Guid><b:Title>CLIMADA v1: A Global Weather and Climate Risk Assessment Platform</b:Title><b:Year>2019</b:Year><b:Author><b:Author><b:NameList><b:Person><b:Last>Aznar-Siguan</b:Last><b:First>Gabriela</b:First></b:Person><b:Person><b:Last>Bresch</b:Last><b:First>David N.</b:First></b:Person></b:NameList></b:Author></b:Author><b:Publisher>Copernicus Publications</b:Publisher><b:Volume>12</b:Volume><b:Pages>3085–3097</b:Pages><b:URL>https://gmd.copernicus.org/articles/12/3085/2019/</b:URL><b:JournalName>Geoscientific Model Development</b:JournalName><b:Comments>DOI: 10.5194/gmd-12-30